In [ ]:
def shoot_for_Wr(a_vars, ddeltaKdr) :
    
    C2 = 1.0
    w0 = C2 / a_vars.r[0] / a_vars.r[0]
    alpha_low = - w0 / a_vars.r[0]
    alpha_high =  w0 / a_vars.r[0]

    root_enclosed = False 
    while root_enclosed == False :
    
        # low end
        y0 = np.array([w0, alpha_low])
        Wr_low, dWrdr_low, Q_low = integrate_using_midpoint(r_vec, dydr_for_Wr, y0, ddeltaKdr)

        # high end  
        y0 = np.array([w0, alpha_high])
        Wr_high, dWrdr_high, Q_high = integrate_using_midpoint(r_vec, dydr_for_Wr, y0, ddeltaKdr)  
    
        # check bounds
        if (Q_low[-1] < 0) and (Q_high[-1] < 0) :
            alpha_high_old = alpha_high
            alpha_high = 10.0 * alpha_high_old
            alpha_low = alpha_high_old
            
        elif ((Q_low[-1] > 0) and (Q_high[-1] > 0)) :
            alpha_low_old = alpha_low
            alpha_low = 10.0 * alpha_low_old
            alpha_high = alpha_low_old
            
        else :
            root_enclosed = True
    
    #print("root now enclosed by alpha ", alpha_low, alpha_high)
    #print("root enclosed by values ", Q_low[-1], Q_high[-1])

    error = 1.0
    count = 0
    while (error > 1.0e-3 and count < 100):
        
        dalpha = (0.0 - Q_low[-1]) * (alpha_high - alpha_low) / (Q_high[-1] - Q_low[-1])
        alpha_test = alpha_low + dalpha
        
        # test new value
        y0 = np.array([w0, alpha_test])
        Wr_test, dWrdr_test, Q_test = integrate_using_midpoint(r_vec, dydr_for_Wr, y0, ddeltaKdr)  
        
        # check error and update interval
        if (Q_test[-1] < 0) :
            alpha_low = alpha_test
            Wr_low, dWrdr_low, Q_low = Wr_test, dWrdr_test, Q_test
            
        else :
            alpha_high = alpha_test
            Wr_high, dWrdr_high, Q_high = Wr_test, dWrdr_test, Q_test
            
        error = min(abs(Q_low[-1]), abs(Q_high[-1]))
        count += 1
        
        #print("root now enclosed by alpha ", alpha_low, alpha_high)
        #print("root enclosed by values ", Q_low[-1], Q_high[-1])
    
    # Return the best values
    if (abs(Q_low[-1]) < abs(Q_high[-1])) :
        
        return Wr_low, dWrdr_low, Q_low
    
    else :
    
        return Wr_high, dWrdr_high, Q_high